# FlightRadar Analytics Project

## 0. Introducción

El objetivo de este proyecto es construir una aplicación de análisis de datos utilizando Apache Spark sobre el dataset que vamos a crear utilizando la API FlightRadar, de la que vamos a sacar información sobre todos los vuelos que han tenido o bien origen o bien destino en los aeropuertos más importantes de España en el último mes.

El dataset va a contener información sobre todos los vuelos que han tenido origen o destino en España en sus aeropuertos más famosos en el último mes. Cada registro representa un vuelo e incluye información sobre aeropuerto de origen, aeropuerto de llegada, aerolínea, información sobre el avión (matrícula, modelo...), día y hora de salida, día y hora de llegada...

La elección de este dataset permite trabajar con un caso de uso muy cercano a un entorno real de Big Data, ya que combina grandes volúmenes de información, datos temporales, datos geográficos, métricas propias del sistema de vuelos. Además, permite aplicar de forma práctica muchas de las funcionalidades principlaes de Apache Spark.

A lo largo del proyecto se seguirá un flujo completo de trabajo, comenzando por la creación del dataset desde datos que vamos a obtener desde la API FlightRadar, análisis del esquema del dataset, la limpieza de datos y las validaciones de calidad. Posteriormente, se enriquecerá el dataset mediante la incorporación de variables derivadas de las ya creadas en el dataset de inicio que nos facilitarán el análisis exploratios, la generación de KPIs y el análisis en tiempo real.

El proyecto también incluirá consultas con DataFrames y Spark SQL, uso de window functions, creación de indicadores de negocio, visualizaciones, simulación de procesamiento en streaming con Structured Streaming.

Con este proyecto se busca no solo analizar vuelos en el últimos mes en los aeropuertos más importantes de España, sino también entender cómo se construye una aplicación Spark completa, desde la ingesta y preparación de los datos hasta la obtención de insights.

%md
## 1. Objetivos

El objetivo principal del proyecto es construir una aplicación completa de análisis de datos con Apache Spark creando nuestro propio dataset utillizando la API FlightRadar sobre en nuestro caso vuelos en el último mes en los aeropuertos más importantes de España.

Los objetivos específicos son:

- Crear nuestro propio DataFrame
- Comprender en profundidad el esquema y significado de las columnas del dataset.
- Limpiar y validar los datos, detectando nulos, duplicados, outliers e inconsistencias.
- Enriquecer el dataset.
- Crear variables derivadas mediante feature engineering.
- Realizar análisis exploratorio con DataFrames y Spark SQL.
- Aplicar Window Functions para análisis avanzados.
- Definir KPIs relevantes del sistema de vuelos en España.
- Construir visualizaciones y dashboards.
- Simular procesamiento en tiempo real con Structured Streaming.
- Documentar limitaciones, aprendizajes y conclusiones del proyecto.

## 2. Creación del DataFrame
Hemos optado por crear un dataset sobre los vuelos en el último mes en los que han participado los aeropuertos más importantes de España. Todo esto con la API FlightRadar como base, a través de esta API podemos obtener información de todos los vuelos que ocurren a través de todo el mundo, aunque como hemos comentado ya varias veces nos hemos centrado en los que han ocurrido en el último mes en los aeropuertos más importantes de España.

Lo que hemos hecho en primer lugar es ejecutar el script que se encuentra en la siguiente ruta: **/Volumes/vuelos_españa/default/scripts/carga_vuelos_ultimo_mes.py**
En este script vamos sacando mediante llamadas a la API FlightRadar día a día de los últimos 30 días un archivo .csv por cada día con la información de los vuelos en los que ha intervenido un aeropuerto de los considerados importantes por nosotros en el último mes. Por lo que tras finalizar correctamente la ejecución del script lo que obtenemos es una carpeta con 30 archivos (uno por cada día) .csv con información de los vuelos que se han llevado a cabo cada día.

Mediante la lectura de estos archivos .csv creamos el DataFrame inicial, en el que tenemos la siguiente información: id del vuelo, número del vuelo, indicativo, icao aerolínea, modelo avión, matrícula avión, icao aeropuerto origen, icao aeropuerto destino, fecha hora despegue, fecha hora aterrizaje. Esta información es muy útil pero consideramos que tener solamente información sobre el icao tanto de aeropuertos y aerolíneas es muy insuficiente por lo que ejecutamos otros 2 scripts para obtener más información sobre aerolíneas y aeropuertos.

El ICAO es un código de 4 letras que identifica de forma única a un aeropuerto o aerolínea a nível internacional. Como nosotros no solo queremos tener este código que individualmente es muy poco descriptivo vamos a sacar más información sobre aerolíneas y aeropuertos.

Ejecutamos el siguiente script: **/Volumes/vuelos_españa/default/scripts/Aerolineas.py**
Con este script lo que hacemos es sacar a un archivo .csv el nombre de la aerolínea al que hace referencía el código ICAO, y su IATA (suele tener 3 letras y es lo que suele ver el pasajero en los billetes, buscadores, etc...) y con esto tenemos un archivo .csv con mucha más información sobre las aerolíneas.

Por último ejecutamos el siguiente script: **/Volumes/vuelos_españa/default/scripts/Aeropuertos.py**
Con este script lo que hacemos es sacar a un archivo .csv el nombre del aeropuerto al que hace referencía el código ICAO, y su IATA (suele tener 3 letras y es lo que suele ver el pasajero en los billetes, buscadores, etc...) y con esto tenemos un archivo .csv con mucha más información sobre los aeropuertos.

Creamos un DataFrame con los datos del fichero .csv sobre las aerolíneas y otro DataFrame con los datos del fichero .csv sobre los aeropuertos. Teniendo estos 3 DataFrames hacemos 3 join left (con ICAO aerolínea, ICAO aeropuerto origen e ICAO aeropuerto destino) y obtenemos el DataFrame completo con el que vamos a comenzar a trabajar.

El DataFrame con el que vamos a comenzar a trabajar tiene las siguientes columnas: `id`, `aeropuerto_origen_nombre`, `eropuerto_origen_icao`, `aeropuerto_origen_iata`, `aeropuerto_destino_nombre`, `aeropuerto_destino_icao`, `aeropuerto_destino_iata`, `aerolinea_nombre`, `aerolinea_icao`, `aerolinea_iata`, `fecha_hora_despegue`, `fecha_hora_aterrizaje`, `numero_vuelo`, `matricula_avion`, `modelo_avion`, `indicativo`

Este DataFrame es mucho más completo que el incial, es mucho más descriptivo y tenemos mayor capacidad de análisis.

In [0]:
df_vuelos_españa = spark.read.format("csv") \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .load("/Volumes/vuelos_españa/default/vuelos/")

In [0]:
df_vuelos_españa.count()

In [0]:
nuevos_nombres = [
    "id",
    "numero_vuelo",
    "indicativo",
    "aerolinea_icao",
    "modelo_avion",
    "matricula_avion",
    "aeropuerto_origen_icao",
    "aeropuerto_destino_icao",
    "fecha_hora_despegue",
    "fecha_hora_aterrizaje"
]

df_vuelos_españa = df_vuelos_españa.toDF(*nuevos_nombres)

In [0]:
display(df_vuelos_españa.limit(100))

In [0]:
from pyspark.sql.functions import col
df_aerolineas = df_vuelos_españa.select(
    col("aerolinea_icao")
).dropna().distinct()

display(df_aerolineas)

Esto es el código icao de la compañia

In [0]:
from pyspark.sql.functions import col

df_icaos_unicos = (
    df_vuelos_españa
    .select(col("aeropuerto_origen_icao").alias("icao"))
    .union(
        df_vuelos_españa.select(col("aeropuerto_destino_icao").alias("icao"))
    )
    .dropna()
    .distinct()
    .orderBy("icao")
)

df_icaos_unicos.display()

In [0]:
df_vuelos_españa_aerolineas = spark.read.format("csv") \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .load("/Volumes/vuelos_españa/default/aerolineas/")

In [0]:
display(df_vuelos_españa_aerolineas)

In [0]:
df_vuelos_españa_aeropuertos = spark.read.format("csv") \
                .option("header", "true") \
                .option("inferSchema", "true") \
                .load("/Volumes/vuelos_españa/default/aeropuertos/")

In [0]:
display(df_vuelos_españa_aeropuertos)

In [0]:
df_aerolineas_join = df_vuelos_españa_aerolineas.select(
    col("icao").alias("aerolinea_icao_join"),
    col("iata").alias("aerolinea_iata"),
    col("name").alias("aerolinea_nombre")
)

df_aeropuertos_origen = df_vuelos_españa_aeropuertos.select(
    col("icao").alias("origen_icao_join"),
    col("iata").alias("aeropuerto_origen_iata"),
    col("name").alias("aeropuerto_origen_nombre")
)

df_aeropuertos_destino = df_vuelos_españa_aeropuertos.select(
    col("icao").alias("destino_icao_join"),
    col("iata").alias("aeropuerto_destino_iata"),
    col("name").alias("aeropuerto_destino_nombre")
)

df_vuelos_españa_completo = (
    df_vuelos_españa
    .join(
        df_aerolineas_join,
        col("aerolinea_icao") == col("aerolinea_icao_join"),
        "left"
    )
    .join(
        df_aeropuertos_origen,
        col("aeropuerto_origen_icao") == col("origen_icao_join"),
        "left"
    )
    .join(
        df_aeropuertos_destino,
        col("aeropuerto_destino_icao") == col("destino_icao_join"),
        "left"
    )
    .drop(
        "aerolinea_icao_join",
        "origen_icao_join",
        "destino_icao_join"
    )
)

df_vuelos_españa_completo.display()

In [0]:
df_vuelos_españa_completo = df_vuelos_españa_completo.select(
    "id",
    "aeropuerto_origen_nombre",
    "aeropuerto_origen_icao",
    "aeropuerto_origen_iata",
    "aeropuerto_destino_nombre",
    "aeropuerto_destino_icao",
    "aeropuerto_destino_iata",
    "aerolinea_nombre",
    "aerolinea_icao",
    "aerolinea_iata",
    "fecha_hora_despegue",
    "fecha_hora_aterrizaje",
    "numero_vuelo",
    "matricula_avion",
    "modelo_avion",
    "indicativo"
)

df_vuelos_españa_completo.display()

## 3. Limpieza, validaciones y Data Quality